**420-C74-SF - Techniques d'apprentissage automatique - Automne 2026 - Spécialiste en solutions d'intelligence artificielle**<br/>
© 2026 MKL.AI - Tous droits réservés
<br/>
![Atelier - Techniques de régularisation](static/12-A-banner.png)
<br/>
**Objectif:** cette séance de travaux pratiques a pour objectif la mise en oeuvre des régularisations de Tikhonov, Lasso et ElasticNet sur le jeu de données **Credit**

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Pour les très petits alpha, Lasso et ElasticNet convergent difficilement : on masque ces avertissements
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

### 1 - Chargement et exploration sommaire des données

**Charger et visualiser le jeu de données Credit**

In [ ]:
import pandas as pd

In [ ]:
credit = pd.read_csv('../../data/Credit.csv', index_col=[0])

In [ ]:
credit.head(10)

In [ ]:
import seaborn as sns; sns.set(style="ticks", color_codes=True)

In [ ]:
g_ = sns.pairplot(credit)

### 2 - Préparation des données

**Préparer le données afin de pouvoir les utiliser dans nos modèles**

In [ ]:
credit_num = credit.copy()
credit_num['Gender'] = (credit['Gender'] == 'Female').astype(int)
credit_num['Student'] = (credit['Student'] == 'Yes').astype(int)
credit_num['Married'] = (credit['Married'] == 'Yes').astype(int)

In [ ]:
credit_num.head()

In [ ]:
X = credit_num.values[:,:-1].astype(float)
y = credit_num.values[:,-1].astype(float)

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
import numpy as np
np.set_printoptions(suppress=True)

In [ ]:
# Ici ne pas élever les variables indicatrices
X_poly = np.c_[X, X[:,:6]**2, X[:,:6]**3, X[:,:6]**4, X[:,:6]**5, X[:,:6]**6]
#X_poly = np.c_[X, X**2, X**3, X**4, X**5, X**6]

In [ ]:
from sklearn import preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_poly)
X_scale = scaler.transform(X_poly)

In [ ]:
X_scale.shape
X_scale

### 3 - Séparation des données train / test

[ sklearn.model_selection.train_test_split(*arrays, **options)](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

**Garder 20% des données pour le test**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scale, y, test_size=0.20, shuffle=True, random_state=2022)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

### 4 - Régression linéaire

[class sklearn.linear_model.LinearRegression(fit_intercept=True, normalize=False, copy_X=True, n_jobs=None)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
coef_reg = reg.coef_
coef_reg

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
y_pred = reg.predict(X_train)
np.sqrt(mean_squared_error(y_train, y_pred))

In [ ]:
y_pred = reg.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

### 5 - Régularisation de Tikhonov - Ridge Regression

[class sklearn.linear_model.RidgeCV(alphas=(0.1, 1.0, 10.0), fit_intercept=True, normalize=False, scoring=None, cv=None, gcv_mode=None, store_cv_values=False)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)

In [ ]:
from sklearn.linear_model import RidgeCV

In [ ]:
ridge = RidgeCV(alphas=np.logspace(-4, 2, 50), cv=5).fit(X_train, y_train)

In [ ]:
ridge.alpha_

**Courbe de validation : erreur en fonction de alpha**

- alpha petit : peu de régularisation, l'erreur d'entraînement est faible mais l'erreur de validation est élevée → **surapprentissage** (variance élevée)
- alpha grand : coefficients trop contraints, les deux erreurs augmentent → **sous-apprentissage** (biais élevé)
- La validation croisée retient l'alpha qui minimise l'erreur de validation

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve

def courbe_validation(modele, alphas, alpha_optimal, titre):
    """Trace le RMSE d'entraînement et de validation croisée en fonction de alpha"""
    train_scores, val_scores = validation_curve(modele, X_train, y_train, param_name='alpha',
                                                param_range=alphas, cv=5,
                                                scoring='neg_root_mean_squared_error')
    plt.figure(figsize=(8, 4))
    plt.semilogx(alphas, -train_scores.mean(axis=1), label='Entraînement')
    plt.semilogx(alphas, -val_scores.mean(axis=1), label='Validation croisée (5 plis)')
    plt.axvline(alpha_optimal, color='gray', linestyle='--', label=f'alpha retenu = {alpha_optimal:.3g}')
    plt.xlabel('alpha (force de la régularisation)')
    plt.ylabel('RMSE')
    plt.title(titre)
    plt.legend()
    plt.show()

In [ ]:
from sklearn.linear_model import Ridge

courbe_validation(Ridge(), np.logspace(-4, 2, 50), ridge.alpha_, 'Ridge')

In [ ]:
coef_ridge = ridge.coef_
coef_ridge

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
modele_final = Ridge(alpha=ridge.alpha_, max_iter=100000).fit(X_train, y_train)

In [ ]:
y_pred = modele_final.predict(X_train)
np.sqrt(mean_squared_error(y_train, y_pred))

In [ ]:
y_pred = modele_final.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

### 6 - Lasso

[class sklearn.linear_model.LassoCV(eps=0.001, n_alphas=100, alphas=None, fit_intercept=True, normalize=False, precompute='auto', max_iter=1000, tol=0.0001, copy_X=True, cv=None, verbose=False, n_jobs=None, positive=False, random_state=None, selection='cyclic')](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html)

In [ ]:
from sklearn.linear_model import LassoCV

In [ ]:
lasso = LassoCV(alphas=np.logspace(-2, 1, 40), max_iter=100000, cv=5).fit(X_train, y_train)

In [ ]:
lasso.alpha_

In [ ]:
from sklearn.linear_model import Lasso

courbe_validation(Lasso(max_iter=100000), np.logspace(-2, 1, 40), lasso.alpha_, 'Lasso')

In [ ]:
lasso.coef_

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
modele_final = Lasso(alpha=lasso.alpha_, max_iter=100000).fit(X_train, y_train)

In [ ]:
modele_final.coef_

In [ ]:
y_pred = modele_final.predict(X_train)
np.sqrt(mean_squared_error(y_train, y_pred))

In [ ]:
y_pred = modele_final.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

### 7 - ElasticNet

[class sklearn.linear_model.ElasticNetCV(l1_ratio=0.5, eps=0.001, n_alphas=100, alphas=None, fit_intercept=True, normalize=False, precompute='auto', max_iter=1000, tol=0.0001, cv=None, copy_X=True, verbose=0, n_jobs=None, positive=False, random_state=None, selection='cyclic')](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNetCV.html)

In [ ]:
from sklearn.linear_model import ElasticNetCV

In [ ]:
en = ElasticNetCV(l1_ratio=0.2, alphas=np.logspace(-5, -1, 40), max_iter=100000, cv=5).fit(X_train, y_train)

In [ ]:
en.alpha_

In [ ]:
from sklearn.linear_model import ElasticNet

courbe_validation(ElasticNet(l1_ratio=0.2, max_iter=100000), np.logspace(-5, -1, 40), en.alpha_, 'ElasticNet (l1_ratio=0.2)')

In [ ]:
en.coef_

In [ ]:
from sklearn.linear_model import ElasticNet

In [ ]:
modele_final = ElasticNet(alpha=en.alpha_, l1_ratio=en.l1_ratio_, max_iter=100000).fit(X_train, y_train)

In [ ]:
y_pred = modele_final.predict(X_train)
np.sqrt(mean_squared_error(y_train, y_pred))

In [ ]:
y_pred = modele_final.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

### 8 - Comparaison avec Ridge et sans Ridge

In [ ]:
def polynomial(X, degree=2):
    vec = [np.ones((X.shape[0],1))]
    for d in range(1,degree+1):
        temp = np.array((X**d))
        vec.append(temp)
    return np.hstack(vec)

In [ ]:
X_poly = np.c_[X, X[:,:6]**2, X[:,:6]**3, X[:,:6]**4, X[:,:6]**5, X[:,:6]**6]

from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = StandardScaler()
scaler.fit(X_poly)
X_scale = scaler.transform(X_poly)

from collections import defaultdict

history = defaultdict(list)

for n in range(1,7):
    X_sub = X_scale[:,0:n*8]
    X_train, X_test, y_train, y_test = train_test_split(X_sub, y, test_size=0.50, random_state=2020)
    reg = LinearRegression().fit(X_train, y_train)
    y_pred = reg.predict(X_train)
    RMSE_train = np.sqrt(mean_squared_error(y_train, y_pred))
    y_pred = reg.predict(X_test)
    RMSE_test = np.sqrt(mean_squared_error(y_test, y_pred))
    history['lr-train'].append(RMSE_train)
    history['lr-test'].append(RMSE_test)
    
    ridge = RidgeCV(alphas=[0.00001, 0.0001, 0.001, 0.003, 0.007, 0.01, 0.05, 0.07], cv=5).fit(X_train, y_train)
    modele_final = Ridge(alpha=ridge.alpha_, max_iter=100000).fit(X_train, y_train)
    y_pred = modele_final.predict(X_train)
    RMSE_train = np.sqrt(mean_squared_error(y_train, y_pred))
    y_pred = modele_final.predict(X_test)
    RMSE_test = np.sqrt(mean_squared_error(y_test, y_pred))
    history['ridge-train'].append(RMSE_train)
    history['ridge-test'].append(RMSE_test)
    
import matplotlib.pyplot as plt
plt.plot(history['ridge-train'], label="Ridge Train")
plt.plot(history['ridge-test'], label="Ridge Test")
plt.plot(history['lr-train'], label="LR Train")
plt.plot(history['lr-test'], label="LR Test")
plt.legend()